In [15]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}

enum class Context { ELIMINATION, BUDGET, CLUSTERING }

val percentageFraction = 1
val colsWithoutPercentages = "strict avg"
val gradient = 0.1

val mode = Mode.RANDOM
val algorithm = Algorithm.OP
val context = Context.BUDGET

val fileName = "comparison_final_fixed.csv"//"comparison_${mode.name.lowercase()}.csv"

val relativePath = "/op-solver-strict/results/" //${context.name.lowercase()}/comparison/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df

instance,strict avg,ref avg,ea avg,strict max,ref max,ea max
eil101,3284,3316,3646,3416,3362,3655
gil262,7198,6882,8104,7601,7353,8105
pr299,8416,7705,8953,8833,8062,9029
lin318,9771,9727,10724,10195,9928,10777
rd400,11244,11314,13318,12382,11397,13426
d493,16092,14913,16780,16816,15355,16822
u574,16811,16159,18913,17423,16506,18936
u724,21043,20589,23441,21771,20669,23621
pcb1173,31675,31360,35055,32632,31664,35613
fl1400,49751,41512,55242,57207,48396,56032


In [16]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Number).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[3655, 8105, 9029, 10777, 13426, 16822, 18936, 23621, 35613, 57207, 70023]

In [17]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, (df[colsWithoutPercentages][index] as Number).toInt()) }

rowMaxValues

[3655, 8105, 9029, 10777, 13426, 16822, 18936, 23621, 35613, 57207, 70023]

In [18]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Int>()
}.map { row -> row!!.toInt()}
rowMinValues

[3284, 6882, 7705, 9727, 11244, 14913, 16159, 20589, 31360, 41512, 63253]

In [19]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Int>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (df.get(colsWithoutPercentages)[row] as Number).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = (df.get(colsWithoutPercentages)[row] as Number).toInt()
        calculatePercentage(refValue, (col[row] as Number).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Number && it.toDouble() > 100.0) {
        "-"
    } else if (it is Number) {
        formatePercentage(it.toDouble())
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -, -4.3\%, +10.3\%, +5.5\%, -1.0\%, +11.0\%]

In [20]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

instance,strict avg,ref avg,ea avg,strict max,ref max,ea max
eil101,\cellcolor{cyan!0} 3284,\cellcolor{cyan!7} 3316{\tiny+1.0\%},\cellcolor{cyan!97} 3646{\tiny+11.0\%},\cellcolor{cyan!34} 3416{\tiny+4.0\%},\cellcolor{cyan!19} 3362{\tiny+2.4\%},\cellcolor{cyan!100} \textbf{3655*}{\...
gil262,\cellcolor{cyan!0} 7198,\cellcolor{cyan!0} 6882{\tiny-4.4\%},\cellcolor{cyan!99} 8104{\tiny+12.6\%},\cellcolor{cyan!37} 7601{\tiny+5.6\%},\cellcolor{cyan!7} 7353{\tiny+2.2\%},\cellcolor{cyan!100} \textbf{8105*}{\...
pr299,\cellcolor{cyan!32} 8416,\cellcolor{cyan!0} 7705{\tiny-8.4\%},\cellcolor{cyan!91} 8953{\tiny+6.4\%},\cellcolor{cyan!78} 8833{\tiny+5.0\%},\cellcolor{cyan!0} 8062{\tiny-4.2\%},\cellcolor{cyan!100} \textbf{9029*}{\...
lin318,\cellcolor{cyan!6} 9771,\cellcolor{cyan!2} 9727{\tiny-0.5\%},\cellcolor{cyan!95} 10724{\tiny+9.8\%},\cellcolor{cyan!45} 10195{\tiny+4.3\%},\cellcolor{cyan!21} 9928{\tiny+1.6\%},\cellcolor{cyan!100} \textbf{10777*}{...
rd400,\cellcolor{cyan!0} 11244,\cellcolor{cyan!0} 11314{\tiny+0.6\%},\cellcolor{cyan!91} 13318{\tiny+18.4\%},\cellcolor{cyan!22} 12382{\tiny+10.1\%},\cellcolor{cyan!0} 11397{\tiny+1.4\%},\cellcolor{cyan!100} \textbf{13426*}{...
d493,\cellcolor{cyan!56} 16092,\cellcolor{cyan!0} 14913{\tiny-7.3\%},\cellcolor{cyan!97} 16780{\tiny+4.3\%},\cellcolor{cyan!99} 16816{\tiny+4.5\%},\cellcolor{cyan!12} 15355{\tiny-4.6\%},\cellcolor{cyan!100} \textbf{16822*}{...
u574,\cellcolor{cyan!0} 16811,\cellcolor{cyan!0} 16159{\tiny-3.9\%},\cellcolor{cyan!98} 18913{\tiny+12.5\%},\cellcolor{cyan!20} 17423{\tiny+3.6\%},\cellcolor{cyan!0} 16506{\tiny-1.8\%},\cellcolor{cyan!100} \textbf{18936*}{...
u724,\cellcolor{cyan!0} 21043,\cellcolor{cyan!0} 20589{\tiny-2.2\%},\cellcolor{cyan!92} 23441{\tiny+11.4\%},\cellcolor{cyan!21} 21771{\tiny+3.5\%},\cellcolor{cyan!0} 20669{\tiny-1.8\%},\cellcolor{cyan!100} \textbf{23621*}{...
pcb1173,\cellcolor{cyan!0} 31675,\cellcolor{cyan!0} 31360{\tiny-1.0\%},\cellcolor{cyan!84} 35055{\tiny+10.7\%},\cellcolor{cyan!16} 32632{\tiny+3.0\%},\cellcolor{cyan!0} 31664{\tiny-0.0\%},\cellcolor{cyan!100} \textbf{35613*}{...
fl1400,\cellcolor{cyan!0} 49751,\cellcolor{cyan!0} 41512{\tiny-16.6\%},\cellcolor{cyan!65} 55242{\tiny+11.0\%},\cellcolor{cyan!100} \textbf{57207*}{...,\cellcolor{cyan!0} 48396{\tiny-2.7\%},\cellcolor{cyan!79} 56032{\tiny+12.6\%}


In [21]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.9cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:elim:${mode.name.lowercase()}"
val title = "emimination methods ${mode.name.lowercase()}."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Emimination method comparison for \$R' = 0.5\$ and \$\\alpha = 0.25\$ with $\\gamma = 0.5\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|  }
                \hline
                \multicolumn{7}{|c|}{emimination methods random.} \\
                \hline
                    instance & strict avg & ref avg & ea avg & strict max & ref max & ea max \\
                \hline
                    eil101 & \cellcolor{cyan!0} 3284 & \cellcolor{cyan!7} 3316{\tiny+1.0\%} & \cellcolor{cyan!97} 3646{\tiny+11.0\%} & \cellcolor{cyan!34} 3416{\tiny+4.0\%} & \cellcolor{cyan!19} 3362{\tiny+2.4\%} & \cellcolor{cyan!100} \textbf{3655*}{\tiny+11.3\%} \\ 
gil262 & \cellcolor{cyan!0} 7198 & \cellcolor{cyan!0} 6882{\tiny-4.4\%} & \cellcolor{cyan!99} 8104{\tiny+12.6\%} & \cellcolor{cyan!37} 7601{\tiny+5.6\%} & \cellcolor{cyan!7} 7353{\tiny+2.2\%} & \cellcolor{cyan!100} \textbf{8105*}{\tiny+12.6\%} \\ 
pr299 & \cellcolor{cyan!32} 8416 & \cellcolor{cyan!0} 7705{\tiny-8.4\%} & \c